# 01 — Data Exploration

Explore the datasets (Dresden / VISION / Synthetic) and visualize
PRNU noise residuals, fingerprints, and device distributions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

from src.prnu_pipeline import PRNUPipeline
from src.utils.helpers import wavelet_denoise
from src.utils.data_loader import extract_patches, load_splits

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Generate synthetic data for exploration
pipeline = PRNUPipeline()

# Create synthetic PRNU pattern
rng = np.random.RandomState(42)
prnu_pattern = rng.normal(0, 0.01, (512, 512, 3))

# Generate a scene
scene = np.random.randint(50, 200, (512, 512, 3), dtype=np.uint8)
scene_f = scene.astype(np.float64) / 255.0

# Apply PRNU model: I = scene * (1 + K) + noise
image = np.clip(scene_f * (1 + prnu_pattern) + rng.normal(0, 0.005, scene_f.shape), 0, 1)
image_uint8 = (image * 255).astype(np.uint8)

# Extract residual
residual = pipeline.extract_noise_residual(image_uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(image_uint8, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[1].imshow(residual[:,:,0], cmap='RdBu', vmin=-0.05, vmax=0.05)
axes[1].set_title('Noise Residual (Ch 0)')
axes[2].imshow(prnu_pattern[:,:,0], cmap='RdBu', vmin=-0.05, vmax=0.05)
axes[2].set_title('Ground Truth PRNU')
plt.tight_layout()
plt.show()

In [ ]:
# Patch extraction demo
patches = extract_patches(residual, patch_size=128)
print(f'Extracted {len(patches)} patches of shape {patches[0].shape}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < len(patches):
        ax.imshow(patches[i][:,:,0], cmap='RdBu', vmin=-0.05, vmax=0.05)
        ax.set_title(f'Patch {i}')
    ax.axis('off')
plt.suptitle('Noise Residual Patches (128x128)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Residual statistics
print(f'Residual shape: {residual.shape}')
print(f'Mean: {residual.mean():.6f}')
print(f'Std:  {residual.std():.6f}')
print(f'Min:  {residual.min():.6f}')
print(f'Max:  {residual.max():.6f}')

plt.figure(figsize=(10, 4))
plt.hist(residual.ravel(), bins=200, density=True, alpha=0.7)
plt.xlabel('Residual Value')
plt.ylabel('Density')
plt.title('Noise Residual Distribution')
plt.xlim(-0.1, 0.1)
plt.show()